In [38]:
import numpy as np

# Читаем матрицу из файла в numpy массив
matrix = np.loadtxt('outputs\matrices\matrix_glina_answer.out')

print(matrix)

[[  0.75  22.5    2.65  57.     1.  ]
 [  4.    25.     2.65  38.     1.  ]
 [  6.    12.5    2.65  -1.8    1.  ]
 [  0.     0.     2.65  -2.     1.  ]
 [  0.     0.     1.   100.     1.  ]]


<>:4: SyntaxWarning: invalid escape sequence '\m'
<>:4: SyntaxWarning: invalid escape sequence '\m'
C:\Users\Алексей\AppData\Local\Temp\ipykernel_58416\2540438067.py:4: SyntaxWarning: invalid escape sequence '\m'
  matrix = np.loadtxt('outputs\matrices\matrix_glina_answer.out')


In [39]:
from pathlib import Path
import lasio as ls

# Путь к LAS файлу (используйте свой путь при необходимости)
las_path = Path("data/las/skv621.las")

# Загружаем данные из LAS в том же стиле, что и в mkm_core.py
lasdata = ls.read(las_path)
keys = list(lasdata.keys())

# Экземпляр кода из функции load_mkm_from_las
def infer_property_mnemonics(
    las_keys,
    prop_mnems=None,
):
    STANDARD_PROP_QUAD = ("POTA", "THOR", "RHOB", "WNKT")
    keyset = set(las_keys)
    if prop_mnems is not None:
        props = tuple(prop_mnems)
        if len(props) != 4:
            raise ValueError(
                "Нужно ровно 4 мнемоники кривых-свойств (после LITO идут 4 столбца + единицы → 5×5)."
            )
        missing = [m for m in props if m not in keyset]
        if missing:
            raise ValueError(
                f"В LAS нет кривых: {missing}. Доступные мнемоники: {sorted(keyset)}"
            )
        return props

    missing_std = [m for m in STANDARD_PROP_QUAD if m not in keyset]
    if missing_std:
        raise ValueError(
            f"Ожидаются кривые {STANDARD_PROP_QUAD}. Не хватает: {missing_std}. "
            f"Доступно: {sorted(keyset)}. Задайте явно: --props M1 M2 M3 M4"
        )
    return STANDARD_PROP_QUAD

depth_mnem = "DEPT"
litho_mnem = "LITO"
prop_mnems = None

keyset = set(keys)
for req, label in ((depth_mnem, "глубины"), (litho_mnem, "литологии")):
    if req not in keyset:
        raise ValueError(
            f"Кривая {req!r} ({label}) отсутствует в LAS. Доступно: {sorted(keyset)}"
        )

props = infer_property_mnemonics(keys, prop_mnems)
curve_order = [depth_mnem, litho_mnem, *props]
data = lasdata.stack_curves(curve_order, sort_curves=False)

litho_raw = np.asarray(data[:, 1], dtype=float).copy()

data = np.c_[data, np.ones(data.shape[0])]
data[data[:, 1] != 1, 1] = 2

is_coll = data[:, 1] == 1
is_glin = data[:, 1] == 2

coll_prop = data[is_coll][:, 2:]
glin_prop = data[is_glin][:, 2:]

print("shape data", data.shape)
print("litho_raw", litho_raw[:10])
print("coll_prop.shape", coll_prop.shape)
print("glin_prop.shape", glin_prop.shape)

# Вывести строку с одной глубины (например, первой)
print("\nОдна строка данных с глубины:")
print("Заголовки:", curve_order + ["ONES"])
print("Данные:", data[0])

shape data (400, 7)
litho_raw [2. 4. 4. 4. 4. 4. 4. 4. 4. 4.]
coll_prop.shape (104, 5)
glin_prop.shape (296, 5)

Одна строка данных с глубины:
Заголовки: ['DEPT', 'LITO', 'POTA', 'THOR', 'RHOB', 'WNKT', 'ONES']
Данные: [2.92000e+03 2.00000e+00 9.35280e+00 4.99784e+01 2.28916e+00 3.99351e+01
 1.00000e+00]


In [5]:
las_example = data[0][2:]
las_example

array([ 9.3528 , 49.9784 ,  2.28916, 39.9351 ,  1.     ])

In [40]:
glin_prop[:, 0] = glin_prop[:, 0] * glin_prop[:, 2]
glin_prop[:, 1] = glin_prop[:, 1] * glin_prop[:, 2]
matrix[:, 0] = matrix[:, 0] * matrix[:, 2]
matrix[:, 1] = matrix[:, 1] * matrix[:, 2]


In [12]:
glin_prop

array([[ 21.41005565, 114.40855414,   2.28916   ,  39.9351    ,
          1.        ],
       [ 20.9930745 , 111.6881009 ,   2.29085   ,  39.7161    ,
          1.        ],
       [ 20.94169533, 111.39210862,   2.29233   ,  39.8784    ,
          1.        ],
       ...,
       [  4.52858877,  13.74595784,   2.67655   ,  14.8277    ,
          1.        ],
       [  4.40348571,  13.25394475,   2.68774   ,  14.9807    ,
          1.        ],
       [  3.90065694,  11.4752398 ,   2.62968   ,  18.6262    ,
          1.        ]], shape=(296, 5))

In [13]:
matrix

array([[  1.9875,  59.625 ,   2.65  ,  57.    ,   1.    ],
       [ 10.6   ,  66.25  ,   2.65  ,  38.    ,   1.    ],
       [ 15.9   ,  33.125 ,   2.65  ,  -1.8   ,   1.    ],
       [  0.    ,   0.    ,   2.65  ,  -2.    ,   1.    ],
       [  0.    ,   0.    ,   1.    , 100.    ,   1.    ]])

In [42]:
mkm = glin_prop @ np.linalg.inv(matrix)
mkm

array([[-4.90850881,  7.74678827, -3.20441753,  1.14744717,  0.21869091],
       [-4.73283522,  7.48417207, -3.0775245 ,  1.10852098,  0.21766667],
       [-4.68504288,  7.41985477, -3.04385173,  1.09227014,  0.2167697 ],
       ...,
       [ 1.63287744, -1.95368573,  1.38316438, -0.04626518, -0.01609091],
       [ 1.75153629, -2.10798974,  1.46333324, -0.08400707, -0.02287273],
       [ 1.83279837, -2.22662946,  1.50064418, -0.11912824,  0.01231515]],
      shape=(296, 5))

In [66]:


# Найти индексы строк, где последнее значение меньше 0
last_col_negative = mkm[:, -1] < 0
mkm_last_col_negative = mkm[last_col_negative]
print("Строки mkm, где последнее значение меньше 0:")
print(mkm_last_col_negative)
print(mkm_last_col_negative.shape)



Строки mkm, где последнее значение меньше 0:
[[ 7.07573671e-01 -2.21235391e-01  9.70209040e-01 -4.14389743e-01
  -4.21575758e-02]
 [ 2.10634675e+00 -2.08894596e+00  1.93412632e+00 -8.97878630e-01
  -5.36484848e-02]
 [ 3.77444944e+00 -4.54041973e+00  3.00177036e+00 -1.20920613e+00
  -2.65939394e-02]
 [ 4.14497907e+00 -5.14316458e+00  3.21172677e+00 -1.15016550e+00
  -6.33757576e-02]
 [ 4.69092194e+00 -5.86382898e+00  3.59627888e+00 -1.27994153e+00
  -1.43430303e-01]
 [ 4.86351460e+00 -6.08701338e+00  3.71550449e+00 -1.35996329e+00
  -1.32042424e-01]
 [ 4.66732721e+00 -5.79725434e+00  3.56643162e+00 -1.36583176e+00
  -7.06727273e-02]
 [ 5.21419819e+00 -6.36858980e+00  3.95817659e+00 -1.67777285e+00
  -1.26012121e-01]
 [ 5.78732015e+00 -7.10509978e+00  4.36333013e+00 -1.83422929e+00
  -2.11321212e-01]
 [ 4.93814927e+00 -6.06327201e+00  3.75109584e+00 -1.44165794e+00
  -1.84315152e-01]
 [ 3.95723407e+00 -4.83971419e+00  3.04101305e+00 -1.06041778e+00
  -9.81151515e-02]
 [ 3.85513069e+00 -4

In [48]:
v = glin_prop[100] @ np.linalg.inv(matrix)
v

array([ 1.86059651, -2.04283019,  1.49011151, -0.3611748 ,  0.05329697])

In [53]:



import numpy as np

def scale_pos_neg_unit_sums(v):
    v = np.asarray(v, dtype=float)
    w = np.zeros_like(v)

    pos = v > 0
    neg = v < 0

    if np.any(pos):
        pos_sum = float(v[pos].sum())
        if pos_sum > 0:
            w[pos] = v[pos] / pos_sum
        else:
            w[pos] = v[pos]

    if np.any(neg):
        neg_sum = float(v[neg].sum())
        if neg_sum < 0:
            w[neg] = v[neg] / (-neg_sum)
        else:
            w[neg] = v[neg]

    return w


# пример


In [56]:
v= [1,2,3,12,-123]
W = scale_pos_neg_unit_sums(v)
print(W)

[ 0.05555556  0.11111111  0.16666667  0.66666667 -1.        ]


In [21]:
v

array([-4.73283522,  7.48417207, -3.0775245 ,  1.10852098,  0.21766667])

In [35]:
v_norm = 20* (v - v.min()) / (v.max() - v.min()) -10
print(v_norm)
print(v_norm.sum())
v_final = v_norm / v_norm.sum()
print(v_final)
print(v_final.sum())

[-10.          10.          -7.2901535   -0.43733697  -1.89572083]
-9.623211297910634
[ 1.03915415 -1.03915415  0.75755933  0.04544605  0.19699462]
1.0


In [37]:
v_norm = (v - -10) / (10 - -10) 
print(v_norm)
print(v_norm.sum())
v_final = v_norm / v_norm.sum()
print(v_final)
print(v_final.sum())

[0.26335824 0.8742086  0.34612378 0.55542605 0.51088333]
2.55
[0.10327774 0.3428269  0.13573481 0.21781414 0.20034641]
1.0


In [20]:
v_norm

array([-1.        ,  1.        , -0.72901535, -0.0437337 , -0.18957208])

In [13]:
# Нужно в матрице matrix первый и второй столбец попарно умножить на значения из третьего столбца
# Предполагается, что matrix имеет форму (n, 3) или больше столбцов
matrix[:, 0] = matrix[:, 0] * matrix[:, 2]
matrix[:, 1] = matrix[:, 1] * matrix[:, 2]
matrix

array([[  3.975,  92.75 ,   2.65 ,  60.   ,   1.   ],
       [ 18.55 , 106.   ,   2.65 ,  50.   ,   1.   ],
       [ 26.5  ,  53.   ,   2.65 ,  -1.8  ,   1.   ],
       [  0.   ,   0.   ,   2.65 ,  -2.   ,   1.   ],
       [  0.   ,   0.   ,   1.   , 100.   ,   1.   ]])

In [14]:
las_example

array([ 9.3528 , 49.9784 ,  2.28916, 39.9351 ,  1.     ])

In [15]:
las_example[0] = las_example[0] * las_example[2]
las_example[1] = las_example[1] * las_example[2]
las_example


array([ 21.41005565, 114.40855414,   2.28916   ,  39.9351    ,
         1.        ])

In [16]:
inv_m_coll = np.linalg.inv(matrix)
mkm = las_example @ inv_m_coll
mkm

array([ 18.52648507, -21.76280828,  13.26291967,  -9.24528736,
         0.21869091])

In [17]:
matrix[:, 0] = matrix[:, 0] / 1000
matrix[:, 1] = matrix[:, 1] / 1000
matrix

array([[ 3.975e-03,  9.275e-02,  2.650e+00,  6.000e+01,  1.000e+00],
       [ 1.855e-02,  1.060e-01,  2.650e+00,  5.000e+01,  1.000e+00],
       [ 2.650e-02,  5.300e-02,  2.650e+00, -1.800e+00,  1.000e+00],
       [ 0.000e+00,  0.000e+00,  2.650e+00, -2.000e+00,  1.000e+00],
       [ 0.000e+00,  0.000e+00,  1.000e+00,  1.000e+02,  1.000e+00]])

In [18]:
las_example[0] = las_example[0] /1000
las_example[1] = las_example[1] /1000
las_example

array([2.14100556e-02, 1.14408554e-01, 2.28916000e+00, 3.99351000e+01,
       1.00000000e+00])

In [21]:
inv_m_coll = np.linalg.inv(matrix)
mkm = las_example @ inv_m_coll
# mkm = inv_m_coll @ las_example
mkm

array([ 18.52648507, -21.76280828,  13.26291967,  -9.24528736,
         0.21869091])

In [ ]:
import numpy as np

# Явно задаём matrix и las_example, чтобы можно было менять значения вручную.
# matrix имеет форму (n, 3) или (n, 4|5...) в зависимости от задачи — демонстрация на (5, 4)
matrix = np.array([
    [1.5, 35.0, 2.65, 60.0,1],
    [7.0, 40.0, 2.65, 50.0,1],
    [10.0, 20.0, 2.65, -1.8,1],
    [0.0, 0.0, 2.65, -2.0,1],
    [0.0, 0.0, 1.0, 100.0,1],
], dtype=float)

# las_example должен быть одним вектором длины как количество столбцов в matrix (здесь 4)
las_example = np.array([9.3528, 49.9784, 2.28916, 39.9351, 1], dtype=float)

# Для удобства можно выводить
print("matrix:\n", matrix)
print("las_example:\n", las_example)

matrix:
 [[  1.5   35.     2.65  60.     1.  ]
 [  7.    40.     2.65  50.     1.  ]
 [ 10.    20.     2.65  -1.8    1.  ]
 [  0.     0.     2.65  -2.     1.  ]
 [  0.     0.     1.   100.     1.  ]]
las_example:
 [ 9.3528  49.9784   2.28916 39.9351   1.     ]


In [26]:
mkm = las_example @ np.linalg.inv(matrix)
mkm

array([ 23.11235103, -27.2431705 ,  16.5386467 , -11.62651814,
         0.21869091])

In [65]:
import numpy as np

# Явно задаём matrix и las_example, чтобы можно было менять значения вручную.
# matrix имеет форму (n, 3) или (n, 4|5...) в зависимости от задачи — демонстрация на (5, 4)
matrix = np.array([
    [0.75, 22.5, 2.65, 57, 1],
    [4, 25, 2.65, 38, 1],
    [6, 12.5, 2.65, -1.8, 1],
    [0.0, 0.0, 2.65, -2.0, 1],
    [0.0, 0.0, 1.0, 100.0, 1],
], dtype=float)

# las_example должен быть одним вектором длины как количество столбцов в matrix (здесь 4)
las_example = np.array([9.3528, 49.9784, 2.28916, 39.9351, 1], dtype=float)

las_example @ np.linalg.inv(matrix) 

array([-6.03017661,  9.40501337, -3.95743684,  1.36390917,  0.21869091])

In [62]:
glin_prop[:, 0] = glin_prop[:, 0] * glin_prop[:, 2]
glin_prop[:, 1] = glin_prop[:, 1] * glin_prop[:, 2]



In [63]:
matrix[:, 0] = matrix[:, 0] * matrix[:, 2]
matrix[:, 1] = matrix[:, 1] * matrix[:, 2]
matrix





array([[  3.975,  92.75 ,   2.65 ,  60.   ,   1.   ],
       [ 18.55 , 106.   ,   2.65 ,  50.   ,   1.   ],
       [ 26.5  ,  53.   ,   2.65 ,  -1.8  ,   1.   ],
       [  0.   ,   0.   ,   2.65 ,  -2.   ,   1.   ],
       [  0.   ,   0.   ,   1.   , 100.   ,   1.   ]])

In [66]:
glin_prop @ np.linalg.inv(matrix) 

array([[-1.66496327e+01,  2.51043574e+01, -1.10866916e+01,
         3.41327597e+00,  2.18690909e-01],
       [-1.61628472e+01,  2.43817285e+01, -1.07352840e+01,
         3.29873603e+00,  2.17666667e-01],
       [-1.60832883e+01,  2.42704455e+01, -1.06796034e+01,
         3.27567649e+00,  2.16769697e-01],
       ...,
       [ 9.00214339e-01, -8.72210390e-01,  1.22371160e+00,
        -2.35624636e-01, -1.60909091e-02],
       [ 1.05791809e+00, -1.08420862e+00,  1.32448027e+00,
        -2.75317008e-01, -2.28727273e-02],
       [ 1.26280942e+00, -1.38547206e+00,  1.41590635e+00,
        -3.05558863e-01,  1.23151515e-02]], shape=(296, 5))

In [67]:
matrix

array([[  0.75,  22.5 ,   2.65,  57.  ,   1.  ],
       [  4.  ,  25.  ,   2.65,  38.  ,   1.  ],
       [  6.  ,  12.5 ,   2.65,  -1.8 ,   1.  ],
       [  0.  ,   0.  ,   2.65,  -2.  ,   1.  ],
       [  0.  ,   0.  ,   1.  , 100.  ,   1.  ]])